In [18]:
import torch, torchvision
from torchvision.models import resnet50, resnet18,  ResNet50_Weights, ResNet18_Weights
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import numpy as np
from tqdm.auto import tqdm
import torch.nn.functional as F
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# QUESTION 1

In [19]:
model_teacher = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

couche_fc = model_teacher.fc
couche_classif = torch.nn.Linear(couche_fc.in_features, 10)
model_teacher.fc = couche_classif
model_teacher = model_teacher.to("cuda")

model_student  = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

couche_fc_student = model_student.fc
couche_classif_student = torch.nn.Linear(couche_fc_student.in_features, 10)
model_student.fc = couche_classif_student
model_student = model_student.to("cuda")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 224MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 212MB/s]


In [20]:
params_teacher = model_teacher.parameters()
total= 0
for i in params_teacher:
    t = i.requires_grad
    if t: 
        total += i.numel()


params_student = model_student.parameters()
total_student = 0
for i in params_student:
    t = i.requires_grad
    if t:
        total_student += i.numel()
print(total_student)

11181642


In [21]:
data_prep = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform= data_prep
)
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform= data_prep
)

train_subset, val_dataset = random_split(
    train_dataset,
    [40000, 10000]
)

train_dataloader = DataLoader(
    train_subset, 
    batch_size=32,
    shuffle=True,
    num_workers=0
)
validation_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle= False,
    num_workers=0
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model_teacher.parameters(),
      lr=0.001
    )

optimizer_student = torch.optim.Adam(
    model_student.parameters(),
    lr=0.001
)

In [ ]:
for epoch in range(10):
    val_loss = 0
    l = 0
    model_teacher.train()
    for batch_X, batch_Y in train_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        
        optimizer.zero_grad()
        logits = model_teacher(batch_X)
        loss = criterion(logits, batch_Y)
        loss.backward()
        optimizer.step()

    model_teacher.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_teacher(batch_X)
            val_loss += criterion(prediction, batch_Y).item()
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = true_label == classes_predites

            l += comparaison.sum().item()

        acc = l / len(val_dataset)

        val_loss = val_loss / len(validation_dataloader)


    print(f"Epoch {epoch + 1} - Validation Loss : {val_loss:.4f}, accuracy: {acc:.2%}")

In [ ]:
torch.save(model_teacher.state_dict(), "/kaggle/working/teacher_resnet50.pth")

In [ ]:
model_teacher.eval()
l_test = 0
test_loss = 0
with torch.no_grad():
    for batch_X, batch_Y in test_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        prediction = model_teacher(batch_X)
        test_loss += criterion(prediction, batch_Y).item()
        true_label = batch_Y
        classes_predites = prediction.argmax(dim=1)
        comparaison = true_label == classes_predites

        l_test += comparaison.sum().item()

    acc = l_test / len(test_dataset)

    test_loss = test_loss / len(test_dataloader)
print(f"Test Loss : {test_loss:.4f}, accuracy: {acc:.2%}")

# QUESTION 2 

In [ ]:
for epoch in range(10):
    val_loss = 0
    l = 0
    model_student.train()
    for batch_X, batch_Y in train_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        
        optimizer_student.zero_grad()
        logits = model_student(batch_X)
        loss = criterion(logits, batch_Y)
        loss.backward()
        optimizer_student.step()

    model_student.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_student(batch_X)
            val_loss += criterion(prediction, batch_Y).item()
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = true_label == classes_predites

            l += comparaison.sum().item()

        acc = l / len(val_dataset)

        val_loss = val_loss / len(validation_dataloader)


    print(f"Epoch {epoch + 1} - Validation Loss : {val_loss:.4f}, accuracy: {acc:.2%}")

In [ ]:
torch.save(model_student.state_dict(), "/kaggle/working/student_resnet18.pth")

In [ ]:
model_student.eval()
l_test = 0
test_loss = 0
with torch.no_grad():
    for batch_X, batch_Y in test_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        prediction = model_student(batch_X)
        test_loss += criterion(prediction, batch_Y).item()
        true_label = batch_Y
        classes_predites = prediction.argmax(dim=1)
        comparaison = true_label == classes_predites

        l_test += comparaison.sum().item()

    acc = l_test / len(test_dataset)

    test_loss = test_loss / len(test_dataloader)
print(f"Test Loss : {test_loss:.4f}, accuracy: {acc:.2%}")

# QUESTION 3

In [6]:
model_teacher.load_state_dict(
    torch.load(
        "/kaggle/input/datasets/romeoagostino/teacher/teacher_resnet50.pth",
        map_location="cuda",
        weights_only=True
    )
)

model_student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model_student.fc = torch.nn.Linear(
    model_student.fc.in_features, 10
)

model_student = model_student.to("cuda")

optimizer_student = torch.optim.Adam(
    model_student.parameters(),
    lr=0.001
)

In [10]:
model_teacher.eval()
for params in model_teacher.parameters():
    params.requires_grad = False
    
criterion = torch.nn.MSELoss()
for epoch in range(10):
    total_mse = 0
    l = 0
    model_student.train()
    
    for batch_X, batch_Y in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/10"
    ):
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
    
        optimizer_student.zero_grad()
        logits = model_student(batch_X)
        with torch.no_grad():
            prediction_teacher = model_teacher(batch_X)
        mse = criterion(logits, prediction_teacher)
        total_mse += mse.item()
        mse.backward()
        optimizer_student.step()

    model_student.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_student(batch_X)
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = batch_Y == classes_predites
            l += comparaison.sum().item()
    
        acc = l / len(val_dataset)
    
    total_mse = total_mse / len(train_dataloader)

    print(f"Epoch {epoch + 1} - Train MSE : {total_mse:.4f}, "f"Validation Accuracy : {acc:.2%}")

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1 - Train MSE : 10.9826, Validation Accuracy : 87.84%


Epoch 2/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2 - Train MSE : 7.2521, Validation Accuracy : 87.23%


Epoch 3/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 3 - Train MSE : 5.9599, Validation Accuracy : 89.02%


... (12 outputs left out)

Epoch 9 - Train MSE : 2.7621, Validation Accuracy : 90.47%


Epoch 10/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>Exception ignored in: 
Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Traceback (most recent call last):
self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers


  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():

AssertionError  File "/usr/lib/python3.13/multiprocessing/proce

Epoch 10 - Train MSE : 2.5249, Validation Accuracy : 89.82%


In [ ]:
model_student.eval()
model_teacher.eval()

correct = 0
test_mse = 0

criterion = torch.nn.MSELoss()

with torch.no_grad():
    for batch_X, batch_Y in tqdm(
        test_dataloader,
        desc="Test Q3"
    ):
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")

        logits_student = model_student(batch_X)
        logits_teacher = model_teacher(batch_X)

        loss = criterion(
            logits_student,
            logits_teacher
        )

        test_mse += loss.item()

        classes_predites = logits_student.argmax(dim=1)

        correct += (
            classes_predites == batch_Y
        ).sum().item()

test_mse /= len(test_dataloader)
test_accuracy = correct / len(test_dataset)

print(f"Q3 - Test MSE : {test_mse:.4f}")
print(f"Q3 - Test Accuracy : {test_accuracy:.2%}")

In [11]:
torch.save(
    model_student.state_dict(),
    f"/kaggle/working/student_kd_epoch_{epoch + 1}.pth"
)

# QUESTION 4 

In [ ]:
model_student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model_student.fc = torch.nn.Linear(
    model_student.fc.in_features, 10
)

model_student = model_student.to("cuda")

model_teacher.eval()

for param in model_teacher.parameters():
    param.requires_grad = False

# Projection des features : 512 -> 2048

projection = torch.nn.Conv2d(
    in_channels=512,
    out_channels=2048,
    kernel_size=1
).to("cuda")

criterion = torch.nn.MSELoss()
lambda_features = 1.0

optimizer = torch.optim.Adam(
    list(model_student.parameters()) +
    list(projection.parameters()),
    lr=0.001
)

In [ ]:
if "hook_student" in globals():
    hook_student.remove()

if "hook_teacher" in globals():
    hook_teacher.remove()

features_student = {}
features_teacher = {}

def recuperer_features_student(module, inputs, output):
    features_student["layer4"] = output

def recuperer_features_teacher(module, inputs, output):
    features_teacher["layer4"] = output

hook_student = model_student.layer4.register_forward_hook(
    recuperer_features_student
)

hook_teacher = model_teacher.layer4.register_forward_hook(
    recuperer_features_teacher
)

In [35]:
for epoch in range(10):

    model_student.train()
    projection.train()

    total_scores = 0
    total_features = 0
    total_loss = 0

    for batch_X, batch_Y in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/10"
    ):
        batch_X = batch_X.to("cuda")

        optimizer.zero_grad()

        logits = model_student(batch_X)

        with torch.no_grad():
            prediction_teacher = model_teacher(batch_X)

        loss_scores = criterion(
            logits,
            prediction_teacher
        )

        features_student_adaptees = projection(
            features_student["layer4"]
        )

        loss_features = criterion(
            features_student_adaptees,
            features_teacher["layer4"]
        )

        loss_total = (
            loss_scores +
            lambda_features * loss_features
        )

        loss_total.backward()
        optimizer.step()

        total_scores += loss_scores.item()
        total_features += loss_features.item()
        total_loss += loss_total.item()

    total_scores /= len(train_dataloader)
    total_features /= len(train_dataloader)
    total_loss /= len(train_dataloader)

    model_student.eval()
    projection.eval()
    l = 0
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
         
            prediction = model_student(batch_X)

            classes_predites = prediction.argmax(dim=1)

            comparaison = batch_Y == classes_predites

            l += comparaison.sum().item()

    acc = l / len(val_dataset)

    print(
        f"Epoch {epoch + 1} - "
        f"Scores MSE : {total_scores:.4f}, "
        f"Features MSE : {total_features:.4f}, "
        f"Total Loss : {total_loss:.4f}, "
        f"Validation Accuracy : {acc:.2%}"
    )

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1 - Scores MSE : 10.0548, Features MSE : 0.1174, Total Loss : 10.1723, Validation Accuracy : 86.39%


Epoch 2/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 2 - Scores MSE : 7.1346, Features MSE : 0.0890, Total Loss : 7.2235, Validation Accuracy : 89.04%


Epoch 3/10:   0%|          | 0/1250 [00:00<?, ?it/s]

... (13 outputs left out)

Epoch 9/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 9 - Scores MSE : 2.7135, Features MSE : 0.0685, Total Loss : 2.7820, Validation Accuracy : 90.31%


Epoch 10/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 10 - Scores MSE : 2.5231, Features MSE : 0.0675, Total Loss : 2.5906, Validation Accuracy : 90.46%


In [37]:
torch.save(
    {
        "epoch": epoch + 1,
        "model_student": model_student.state_dict(),
        "projection": projection.state_dict(),
        "optimizer": optimizer.state_dict(),
        "validation_accuracy": acc,
    },
    "/kaggle/working/student_kd_scores_features_checkpoint.pth"
)

# QUESTION 5

In [40]:
model_student_scratch = resnet18(weights=None)

couche_fc_student = model_student_scratch.fc
couche_classif_student = torch.nn.Linear(couche_fc_student.in_features, 10)
model_student_scratch.fc = couche_classif_student
model_student_scratch = model_student_scratch.to("cuda")

optimizer_scratch = torch.optim.Adam(
    model_student_scratch.parameters(),
    lr=0.001
)

criterion_scratch = torch.nn.CrossEntropyLoss()

In [42]:
for epoch in range(10):
    val_loss = 0
    l = 0

    model_student_scratch.train()

    for batch_X, batch_Y in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/10"
    ):
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")

        optimizer_scratch.zero_grad()
        logits = model_student_scratch(batch_X)
        loss = criterion_scratch(logits, batch_Y)
        loss.backward()
        optimizer_scratch.step()

    model_student_scratch.eval()

    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")

            prediction = model_student_scratch(batch_X)

            val_loss += criterion_scratch(prediction, batch_Y).item()

            classes_predites = prediction.argmax(dim=1)
            comparaison = batch_Y == classes_predites
            l += comparaison.sum().item()

    acc = l / len(val_dataset)
    val_loss = val_loss / len(validation_dataloader)

    print(
        f"Epoch {epoch + 1} - "
        f"Validation Loss : {val_loss:.4f}, "
        f"Accuracy : {acc:.2%}"
    )

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1 - Validation Loss : 0.8280, Accuracy : 71.04%


Epoch 2/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 2 - Validation Loss : 0.6282, Accuracy : 78.18%


Epoch 3/10:   0%|          | 0/1250 [00:00<?, ?it/s]

... (13 outputs left out)

Epoch 9 - Validation Loss : 0.6465, Accuracy : 83.64%


Epoch 10/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 10 - Validation Loss : 0.7070, Accuracy : 82.62%


In [43]:
model_student_scratch.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for batch_X, batch_Y in tqdm(
        test_dataloader,
        desc="Test Q5"
    ):
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")

        logits = model_student_scratch(batch_X)

        loss = criterion_scratch(logits, batch_Y)
        test_loss += loss.item()

        classes_predites = logits.argmax(dim=1)
        correct += (classes_predites == batch_Y).sum().item()

test_loss /= len(test_dataloader)
test_accuracy = correct / len(test_dataset)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.2%}")

Epoch 10/10:   0%|          | 0/313 [00:00<?, ?it/s]

Test Loss : 0.7473
Test Accuracy : 82.04%


In [44]:
torch.save(
    model_student_scratch.state_dict(),
    "/kaggle/working/student_resnet18_scratch.pth"
)

# QUESTION 6

In [45]:
model_student_kd_scratch = resnet18(weights=None)

model_student_kd_scratch.fc = torch.nn.Linear(
    model_student_kd_scratch.fc.in_features,
    10
)

model_student_kd_scratch = model_student_kd_scratch.to("cuda")

In [47]:
model_teacher.eval()

for param in model_teacher.parameters():
    param.requires_grad = False

criterion_kd_scratch = torch.nn.MSELoss()

optimizer_kd_scratch = torch.optim.Adam(
    model_student_kd_scratch.parameters(),
    lr=0.001
)

In [48]:
for epoch in range(10):
    model_student_kd_scratch.train()

    total_loss = 0

    for batch_X, batch_Y in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/10"
    ):
        batch_X = batch_X.to("cuda")

        optimizer_kd_scratch.zero_grad()

        logits_student = model_student_kd_scratch(batch_X)

        with torch.no_grad():
            logits_teacher = model_teacher(batch_X)

        loss = criterion_kd_scratch(
            logits_student,
            logits_teacher
        )

        loss.backward()
        optimizer_kd_scratch.step()

        total_loss += loss.item()

    total_loss /= len(train_dataloader)

    model_student_kd_scratch.eval()
    correct = 0
    
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
    
            logits = model_student_kd_scratch(batch_X)
            classes_predites = logits.argmax(dim=1)
    
            correct += (classes_predites == batch_Y).sum().item()
    
    acc = correct / len(val_dataset)


    print(
        f"Epoch {epoch + 1} - "
        f"MSE : {total_loss:.4f}, "
        f"Validation Accuracy : {acc:.2%}"
    )

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 1 - MSE : 31.8230, Validation Accuracy : 58.96%


Epoch 2/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2 - MSE : 15.0366, Validation Accuracy : 73.03%


Epoch 3/10:   0%|          | 0/1250 [00:00<?, ?it/s]

... (13 outputs left out)

Epoch 9 - MSE : 3.6779, Validation Accuracy : 87.87%


Epoch 10/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>
: Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a80df38d940>

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python3.13/multiprocessing/process.py", line

Epoch 10 - MSE : 3.2078, Validation Accuracy : 89.19%


In [49]:
model_student_kd_scratch.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for batch_X, batch_Y in tqdm(test_dataloader, desc="Test Q6"):
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")

        logits_student = model_student_kd_scratch(batch_X)
        logits_teacher = model_teacher(batch_X)

        loss = criterion_kd_scratch(
            logits_student,
            logits_teacher
        )

        test_loss += loss.item()

        classes_predites = logits_student.argmax(dim=1)
        correct += (classes_predites == batch_Y).sum().item()

test_loss /= len(test_dataloader)
test_accuracy = correct / len(test_dataset)

print(f"Test MSE : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.2%}")

Test Q6:   0%|          | 0/313 [00:00<?, ?it/s]

Test MSE : 6.1178
Test Accuracy : 85.93%


In [50]:
torch.save(
    model_student_kd_scratch.state_dict(),
    "/kaggle/working/student_kd_scratch_scores.pth"
)

# QUESTION 7

In [51]:
model_student_kd_features_scratch = resnet18(weights=None)

model_student_kd_features_scratch.fc = torch.nn.Linear(
    model_student_kd_features_scratch.fc.in_features,
    10
)

model_student_kd_features_scratch = model_student_kd_features_scratch.to("cuda")

In [52]:
projection_kd_scratch = torch.nn.Conv2d(
    in_channels=512,
    out_channels=2048,
    kernel_size=1
).to("cuda")

features_student_scratch = {}
features_teacher_scratch = {}

def hook_student_scratch(module, inputs, output):
    features_student_scratch["layer4"] = output

def hook_teacher_scratch(module, inputs, output):
    features_teacher_scratch["layer4"] = output

handle_student_scratch = model_student_kd_features_scratch.layer4.register_forward_hook(
    hook_student_scratch
)

handle_teacher_scratch = model_teacher.layer4.register_forward_hook(
    hook_teacher_scratch
)

In [53]:
model_teacher.eval()

for param in model_teacher.parameters():
    param.requires_grad = False

criterion_kd_features_scratch = torch.nn.MSELoss()
lambda_features_scratch = 1.0

optimizer_kd_features_scratch = torch.optim.Adam(
    list(model_student_kd_features_scratch.parameters()) +
    list(projection_kd_scratch.parameters()),
    lr=0.001
)

In [ ]:
for epoch in range(10):

    model_student_kd_features_scratch.train()
    projection_kd_scratch.train()

    total_loss = 0
    total_scores = 0
    total_features = 0

    for batch_X, batch_Y in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch + 1}/10"
    ):
        batch_X = batch_X.to("cuda")

        optimizer_kd_features_scratch.zero_grad()

        logits_student = model_student_kd_features_scratch(batch_X)

        with torch.no_grad():
            logits_teacher = model_teacher(batch_X)

        loss_scores = criterion_kd_features_scratch(
            logits_student,
            logits_teacher
        )

        features_student_adaptees = projection_kd_scratch(
            features_student_scratch["layer4"]
        )

        loss_features = criterion_kd_features_scratch(
            features_student_adaptees,
            features_teacher_scratch["layer4"]
        )

        loss = (
            loss_scores +
            lambda_features_scratch * loss_features
        )

        loss.backward()
        optimizer_kd_features_scratch.step()

        total_loss += loss.item()
        total_scores += loss_scores.item()
        total_features += loss_features.item()

    total_loss /= len(train_dataloader)
    total_scores /= len(train_dataloader)
    total_features /= len(train_dataloader)

    model_student_kd_features_scratch.eval()
    projection_kd_scratch.eval()

    correct = 0

    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")

            logits = model_student_kd_features_scratch(batch_X)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    acc = correct / len(val_dataset)

    print(
        f"Epoch {epoch + 1} - "
        f"Scores MSE : {total_scores:.4f}, "
        f"Features MSE : {total_features:.4f}, "
        f"Total Loss : {total_loss:.4f}, "
        f"Validation Accuracy : {acc:.2%}"
    )

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1 - Scores MSE : 32.4445, Features MSE : 0.2205, Total Loss : 32.6649, Validation Accuracy : 64.11%


Epoch 2/10:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2 - Scores MSE : 15.5127, Features MSE : 0.1413, Total Loss : 15.6539, Validation Accuracy : 75.44%


Epoch 3/10:   0%|          | 0/1250 [00:00<?, ?it/s]

In [ ]:
torch.save(
    {
        "epoch": epoch + 1,
        "model_student": model_student_kd_features_scratch.state_dict(),
        "projection": projection_kd_scratch.state_dict(),
        "validation_accuracy": acc,
    },
    "/kaggle/working/student_kd_features_scratch.pth"
)

# QUESTION 8 - dans le rapport

# QUESTION 9 

In [ ]:
model_teacher.eval()

for param in model_teacher.parameters():
    param.requires_grad = False

temperature = 2.0

def loss_q9(logits_student, logits_teacher, methode):

    if methode == "L1":
        return F.l1_loss(
            logits_student,
            logits_teacher
        )

    elif methode == "KL":
        return F.kl_div(
            F.log_softmax(logits_student / temperature, dim=1),
            F.softmax(logits_teacher / temperature, dim=1),
            reduction="batchmean"
        ) * (temperature ** 2)

    else:
        raise ValueError("Méthode inconnue")

In [ ]:
def entrainer_q9(methode, epochs=10):

    model_q9 = resnet18(
        weights=ResNet18_Weights.IMAGENET1K_V1
    )

    model_q9.fc = torch.nn.Linear(
        model_q9.fc.in_features,
        10
    )

    model_q9 = model_q9.to(device)

    optimizer_q9 = torch.optim.Adam(
        model_q9.parameters(),
        lr=0.001
    )

    for epoch in range(epochs):

        model_q9.train()
        total_loss = 0

        for batch_X, batch_Y in tqdm(
            train_dataloader,
            desc=f"{methode} - Epoch {epoch + 1}/{epochs}"
        ):
            batch_X = batch_X.to(device)

            optimizer_q9.zero_grad()

            logits_student = model_q9(batch_X)

            with torch.no_grad():
                logits_teacher = model_teacher(batch_X)

            loss = loss_q9(
                logits_student,
                logits_teacher,
                methode
            )

            loss.backward()
            optimizer_q9.step()

            total_loss += loss.item()

        total_loss /= len(train_dataloader)

        model_q9.eval()
        correct = 0

        with torch.no_grad():
            for batch_X, batch_Y in validation_dataloader:
                batch_X = batch_X.to(device)
                batch_Y = batch_Y.to(device)

                logits = model_q9(batch_X)

                correct += (
                    logits.argmax(dim=1) == batch_Y
                ).sum().item()

        val_accuracy = correct / len(val_dataset)

        print(
            f"{methode} - Epoch {epoch + 1} - "
            f"Loss : {total_loss:.4f}, "
            f"Validation Accuracy : {val_accuracy:.2%}"
        )

    model_q9.eval()
    correct = 0

    with torch.no_grad():
        for batch_X, batch_Y in tqdm(
            test_dataloader,
            desc=f"Test {methode}"
        ):
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            logits = model_q9(batch_X)

            correct += (
                logits.argmax(dim=1) == batch_Y
            ).sum().item()

    test_accuracy = correct / len(test_dataset)

    print(
        f"{methode} - Test Accuracy : {test_accuracy:.2%}"
    )

    torch.save(
        model_q9.state_dict(),
        f"/kaggle/working/student_q9_{methode}.pth"
    )

    return test_accuracy

In [ ]:
resultats_q9 = {}

for methode in ["L1", "KL"]:

    accuracy = entrainer_q9(
        methode=methode,
        epochs=10
    )

    resultats_q9[methode] = accuracy

print("\n=== COMPARAISON QUESTION 9 ===")

for methode, accuracy in resultats_q9.items():
    print(f"{methode} : {accuracy:.2%}")

# QUESTION 10 

In [22]:
def creer_cnn_custom(compression=1):

    canaux = [
        max(8, round(c / compression))
        for c in [64, 128, 256, 512]
    ]

    couches = []
    entree = 3
    for sortie in canaux:
        couches.extend([
            nn.Conv2d(entree, sortie, kernel_size=3, padding=1),
            nn.BatchNorm2d(sortie),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ])
        entree = sortie
    
    model = nn.Sequential(
        *couches,
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(canaux[-1], 10)
    )
    return model


def forward_cnn_custom(model, batch_X, return_features=False):

    features = model[:16](batch_X)
    logits = model[16:](features)

    if return_features:
        return logits, features

    return logits

In [23]:
def compter_parametres(model):
    return sum(p.numel() for p in model.parameters())

teacher_custom = creer_cnn_custom(compression=1).to(device)

for facteur in [2, 3, 4]:
    student = creer_cnn_custom(compression=facteur)
    n = compter_parametres(student)

    print(
        f"Student /{facteur} : {n} paramètres, "
        f"compression réelle : "
        f"{compter_parametres(teacher_custom) / n:.2f}x"
    )

Student /2 : 391946 paramètres, compression réelle : 3.98x
Student /3 : 175084 paramètres, compression réelle : 8.90x
Student /4 : 99210 paramètres, compression réelle : 15.70x


In [24]:
def entrainer_supervise(model, epochs=10):

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for batch_X, batch_Y in tqdm(
            train_dataloader,
            desc=f"Epoch {epoch+1}/{epochs}"
        ):
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)
            loss = criterion(logits, batch_Y)

            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0

        with torch.no_grad():
            for batch_X, batch_Y in validation_dataloader:
                batch_X = batch_X.to(device)
                batch_Y = batch_Y.to(device)

                logits = model(batch_X)
                correct += (
                    logits.argmax(dim=1) == batch_Y
                ).sum().item()

        accuracy = correct / len(val_dataset)

        print(
            f"Epoch {epoch+1} - "
            f"Validation Accuracy : {accuracy:.2%}"
        )

In [25]:
def evaluer_test(model):
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch_X, batch_Y in test_dataloader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            logits = model(batch_X)

            correct += (
                logits.argmax(dim=1) == batch_Y
            ).sum().item()

    return correct / len(test_dataset)

In [26]:
def entrainer_kd_custom(student, teacher, mode="scores", epochs=3):

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    criterion = nn.MSELoss()

    projection = None
    parametres = list(student.parameters())

    if mode == "scores_features":
        projection = nn.Conv2d(
            student[12].out_channels,
            teacher[12].out_channels,
            kernel_size=1
        ).to(device)

        parametres += list(projection.parameters())

    optimizer = torch.optim.Adam(parametres, lr=0.001)

    for epoch in range(epochs):
        student.train()

        if projection is not None:
            projection.train()

        total_loss = 0

        for batch_X, batch_Y in tqdm(
            train_dataloader,
            desc=f"KD {mode} - Epoch {epoch+1}/{epochs}"
        ):
            batch_X = batch_X.to(device)

            optimizer.zero_grad()

            logits_s, features_s = forward_cnn_custom(
                student, batch_X, return_features=True
            )

            with torch.no_grad():
                logits_t, features_t = forward_cnn_custom(
                    teacher, batch_X, return_features=True
                )

            loss = criterion(logits_s, logits_t)

            if projection is not None:
                loss_features = criterion(
                    projection(features_s),
                    features_t
                )

                loss = loss + loss_features

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        student.eval()
        correct = 0

        with torch.no_grad():
            for batch_X, batch_Y in validation_dataloader:
                batch_X = batch_X.to(device)
                batch_Y = batch_Y.to(device)

                logits = student(batch_X)

                correct += (
                    logits.argmax(dim=1) == batch_Y
                ).sum().item()

        accuracy = correct / len(val_dataset)

        print(
            f"Epoch {epoch+1} - "
            f"Loss : {total_loss / len(train_dataloader):.4f}, "
            f"Validation Accuracy : {accuracy:.2%}"
        )

    return projection

In [29]:
resultats_q10 = []

for facteur in [2, 4]:

    for methode in ["supervised", "scores"]:

        student = creer_cnn_custom(compression=facteur).to(device)

        projection = None

        if methode == "supervised":
            entrainer_supervise(student, epochs=3)
        else:
            projection = entrainer_kd_custom(
                student,
                teacher_custom,
                mode=methode,
                epochs=3
            )

        accuracy_test = evaluer_test(student)

        resultat = {
            "compression": facteur,
            "methode": methode,
            "parametres": compter_parametres(student),
            "accuracy_test": accuracy_test
        }

        resultats_q10.append(resultat)

        checkpoint = {
            "student": student.state_dict()
        }

        if projection is not None:
            checkpoint["projection"] = projection.state_dict()

        torch.save(
            checkpoint,
            f"/kaggle/working/q10_student_{facteur}_{methode}.pth"
        )

        print(
            f"Student /{facteur}, {methode} : "
            f"{accuracy_test:.2%}"
        )
        

Epoch 1/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1 - Validation Accuracy : 46.94%


Epoch 2/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2 - Validation Accuracy : 51.27%


Epoch 3/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 3 - Validation Accuracy : 54.30%
Student /2, supervised : 55.26%


... (14 outputs left out)

KD scores - Epoch 2/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2 - Loss : 0.0000, Validation Accuracy : 12.97%


KD scores - Epoch 3/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 3 - Loss : 0.0000, Validation Accuracy : 13.24%
Student /4, scores : 13.38%
